# L21: RAG 完整流程实现

> 将检索和生成整合，构建完整的 RAG 系统

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.rag.pipeline import RAGPipeline
print("✓ 模块导入成功")

## 21.1 RAG 完整流程

### 端到端架构

```
┌─────────────────────────────────────────────────────────────┐
│                         RAG Pipeline                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 文档处理阶段 (离线)                                       │
│     文档 → 分割 → Embedding → 向量库存储                     │
│                                                             │
│  2. 检索阶段 (在线)                                          │
│     用户问题 → Embedding → 向量检索                         │
│                                                             │
│  3. 生成阶段 (在线)                                          │
│     检索结果 + 问题 → LLM → 回答                             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 查看项目中的 RAG Pipeline

In [ ]:
import inspect
from backend.rag.pipeline import RAGPipeline

print("=== RAGPipeline 源码（部分）===\n")
source = inspect.getsource(RAGPipeline)
print(source[:2000] + "\n... (源码已截断)")

## 21.2 实现 RAG Prompt 模板

In [ ]:
# RAG Prompt 模板
RAG_PROMPT_TEMPLATE = """你是一个有用的助手。请根据以下上下文回答用户的问题。

上下文:
{context}

问题: {question}

要求:
1. 只使用上下文中的信息回答
2. 如果上下文中没有相关信息，请明确说明
3. 回答要准确、简洁
4. 可以引用上下文中的具体内容

回答:"""

print("=== RAG Prompt 模板 ===\n")
print(RAG_PROMPT_TEMPLATE)

## 21.3 简单 RAG 实现

In [ ]:
import numpy as np

class SimpleRAG:
    """
    简单的 RAG 实现
    """
    
    def __init__(self, llm=None):
        self.llm = llm
        self.documents = []  # 存储文档
        self.embeddings = []  # 存储向量
    
    def add_document(self, content: str, embedding: List[float] = None):
        """添加文档"""
        if embedding is None:
            # 使用 mock embedding
            np.random.seed(hash(content) % 10000)
            emb = np.random.randn(384)
            emb = emb / np.linalg.norm(emb)
            embedding = emb.tolist()
        
        self.documents.append(content)
        self.embeddings.append(np.array(embedding))
    
    def retrieve(self, query_embedding: List[float], top_k: int = 3) -> List[str]:
        """检索相关文档"""
        if not self.documents:
            return []
        
        query_vec = np.array(query_embedding)
        
        # 计算相似度
        similarities = []
        for emb in self.embeddings:
            sim = np.dot(query_vec, emb) / (np.linalg.norm(query_vec) * np.linalg.norm(emb))
            similarities.append(sim)
        
        # 排序并返回 top-k
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [self.documents[i] for i in top_indices]
    
    def generate_prompt(self, question: str, context: List[str]) -> str:
        """生成 RAG Prompt"""
        context_str = "\n".join([f"- {doc}" for doc in context])
        return RAG_PROMPT_TEMPLATE.format(
            context=context_str,
            question=question
        )
    
    async def query(self, question: str, query_embedding: List[float] = None) -> str:
        """
        RAG 查询
        
        Args:
            question: 用户问题
            query_embedding: 问题的 Embedding（可选）
            
        Returns:
            回答
        """
        # 1. 生成查询向量（如果没提供）
        if query_embedding is None:
            np.random.seed(hash(question) % 10000)
            emb = np.random.randn(384)
            emb = emb / np.linalg.norm(emb)
            query_embedding = emb.tolist()
        
        # 2. 检索相关文档
        relevant_docs = self.retrieve(query_embedding, top_k=3)
        
        if not relevant_docs:
            return "抱歉，我没有找到相关信息。"
        
        # 3. 生成 Prompt
        prompt = self.generate_prompt(question, relevant_docs)
        
        # 4. 调用 LLM（模拟）
        if self.llm:
            response = await self.llm.chat([Message(role="user", content=prompt)])
            return response.content
        else:
            # 模拟回答
            return f"根据上下文，{relevant_docs[0][:50]}..."

# 创建 RAG 实例
rag = SimpleRAG()

# 添加文档
docs = [
    "Python 是一种高级编程语言，由 Guido van Rossum 创建。",
    "Python 广泛应用于 Web 开发、数据科学、人工智能等领域。",
    "Django 和 Flask 是流行的 Python Web 框架。",
    "NumPy 和 Pandas 是 Python 数据科学的核心库。",
    "TensorFlow 和 PyTorch 是 Python 深度学习框架。",
]

for doc in docs:
    rag.add_document(doc)

print(f"已添加 {len(docs)} 个文档到 RAG 系统")

## 21.4 RAG 查询演示

In [ ]:
async def demo_rag_query():
    """演示 RAG 查询"""
    print("=== RAG 查询演示 ===\n")
    
    questions = [
        "Python 是谁创建的？",
        "Python 有哪些 Web 框架？",
        "数据科学常用的库是什么？",
        "深度学习框架有哪些？",
    ]
    
    for question in questions:
        print(f"问题: {question}")
        
        # 模拟查询
        answer = await rag.query(question)
        
        print(f"回答: {answer}\n")

await demo_rag_query()

## 21.5 高级 RAG 技巧

### 1. 查询重写 (Query Rewriting)

优化用户查询以提高检索质量

In [ ]:
class QueryRewriter:
    """
    查询重写器
    """
    
    @staticmethod
    async def rewrite(query: str, method: str = "expand") -> List[str]:
        """
        重写查询
        
        Args:
            query: 原始查询
            method: 重写方法 (expand/clarify/decompose)
            
        Returns:
            重写后的查询列表
        """
        if method == "expand":
            # 查询扩展：添加同义词
            synonyms = {
                "Python": ["Python语言", "Py"],
                "创建": ["开发", "发明", "设计"],
                "框架": ["库", "工具", "平台"],
            }
            
            queries = [query]
            for word, syns in synonyms.items():
                if word in query:
                    for syn in syns:
                        queries.append(query.replace(word, syn))
            
            return queries
        
        elif method == "decompose":
            # 查询分解：复杂问题拆分成多个子问题
            if "和" in query or "和" in query:
                return [q.strip() for q in query.replace("和", "").replace("和", "|").split("|")]
            return [query]
        
        return [query]

# 演示查询重写
async def demo_query_rewrite():
    print("=== 查询重写演示 ===\n")
    
    queries = [
        "Python 的框架和库",
        "谁创建了 Python",
    ]
    
    for q in queries:
        print(f"原始查询: {q}")
        
        expanded = await QueryRewriter.rewrite(q, method="expand")
        print(f"扩展后: {expanded}")
        
        decomposed = await QueryRewriter.rewrite(q, method="decompose")
        print(f"分解后: {decomposed}")
        print()

await demo_query_rewrite()

### 2. HyDE (Hypothetical Document Embeddings)

让 LLM 先生成理想答案，然后用这个答案来检索

In [ ]:
async def hyde_retrieval(query: str, rag: SimpleRAG) -> str:
    """
    HyDE 检索流程
    """
    print(f"原始查询: {query}")
    
    # 1. 让 LLM 生成理想答案
    hypothetical_answer = f"关于 {query} 的详细解答，包括定义、特点、应用等。"
    print(f"假设答案: {hypothetical_answer[:50]}...")
    
    # 2. 用假设答案的 Embedding 检索
    np.random.seed(hash(hypothetical_answer) % 10000)
    emb = np.random.randn(384)
    emb = emb / np.linalg.norm(emb)
    
    relevant_docs = rag.retrieve(emb.tolist())
    print(f"检索到 {len(relevant_docs)} 个相关文档")
    
    # 3. 用检索结果和原始查询生成最终答案
    # ...
    
    return f"基于 HyDE 的回答"

# 演示
await hyde_retrieval("Python 的应用领域", rag)

## 21.6 RAG 系统评估

### 评估维度

In [ ]:
class RAGEvaluator:
    """
    RAG 系统评估器
    """
    
    @staticmethod
    def evaluate_retrieval(
        retrieved_docs: List[str],
        relevant_docs: List[str]
    ) -> Dict[str, float]:
        """
        评估检索质量
        """
        retrieved_set = set(retrieved_docs)
        relevant_set = set(relevant_docs)
        
        intersection = retrieved_set & relevant_set
        
        precision = len(intersection) / len(retrieved_set) if retrieved_set else 0
        recall = len(intersection) / len(relevant_set) if relevant_set else 0
        
        return {
            "precision": precision,
            "recall": recall,
            "f1": 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        }
    
    @staticmethod
    def evaluate_generation(
        answer: str,
        ground_truth: str
    ) -> Dict[str, float]:
        """
        评估生成质量（简化版）
        """
        # 简单的字符级别相似度
        common_chars = set(answer.lower()) & set(ground_truth.lower())
        total_chars = set(answer.lower()) | set(ground_truth.lower())
        
        char_similarity = len(common_chars) / len(total_chars) if total_chars else 0
        
        # 包含关键词检查
        keywords_in_answer = sum(1 for kw in ground_truth.split() if kw.lower() in answer.lower())
        keyword_coverage = keywords_in_answer / len(ground_truth.split()) if ground_truth.split() else 0
        
        return {
            "char_similarity": char_similarity,
            "keyword_coverage": keyword_coverage
        }

# 演示评估
def demo_evaluation():
    print("=== RAG 评估演示 ===\n")
    
    evaluator = RAGEvaluator()
    
    # 评估检索
    retrieved = ["doc1", "doc3", "doc5"]
    relevant = ["doc1", "doc3", "doc4", "doc7"]
    
    retrieval_metrics = evaluator.evaluate_retrieval(retrieved, relevant)
    print("检索评估:")
    print(f"  精确率: {retrieval_metrics['precision']:.2%}")
    print(f"  召回率: {retrieval_metrics['recall']:.2%}")
    print(f"  F1 分数: {retrieval_metrics['f1']:.2%}\n")
    
    # 评估生成
    answer = "Python 由 Guido van Rossum 创建，广泛应用于 Web 开发和数据分析。"
    ground_truth = "Python 是由 Guido van Rossum 创建的编程语言"
    
    generation_metrics = evaluator.evaluate_generation(answer, ground_truth)
    print("生成评估:")
    print(f"  字符相似度: {generation_metrics['char_similarity']:.2%}")
    print(f"  关键词覆盖: {generation_metrics['keyword_coverage']:.2%}")

demo_evaluation()

## 21.7 常见面试问题

**Q: RAG 系统的核心挑战是什么？**
- A:
  1. **检索质量**：找到真正相关的文档
  2. **上下文长度**：检索结果可能很长
  3. **答案准确性**：避免幻觉和错误信息
  4. **延迟问题**：检索 + 生成需要时间
  5. **知识更新**：向量库需要同步更新

**Q: 如何提升 RAG 的检索准确率？**
- A:
  1. **优化文档分割**：保持语义完整
  2. **混合检索**：向量 + 关键词
  3. **查询重写**：扩展和优化查询
  4. **重排序**：对结果重新评分
  5. **调整参数**：top_k、阈值等

**Q: RAG 和 Fine-tuning 有什么区别？**
- A:
  | 特性 | RAG | Fine-tuning |
  |------|-----|-------------|
  | 知识更新 | 简单，更新文档即可 | 需要重新训练 |
  | 成本 | 低 | 高 |
  | 知识来源 | 外部文档 | 内化到模型 |
  | 适用场景 | 知识密集、变化快 | 特定任务、风格 |
  | 可解释性 | 可追溯来源 | 不可解释 |
  | 推荐方案 | 先用 RAG，再考虑 Fine-tune |

**Q: 如何处理 RAG 的幻觉问题？**
- A:
  1. **严格限制**：要求只使用检索结果
  2. **引用标注**：要求 LLM 标注信息来源
  3. **验证机制**：检查答案是否基于检索结果
  4. **置信度评分**：对答案的可信度打分
  5. **人工审核**：重要问题人工验证

---

## 总结

本课程学习了 RAG 完整流程的核心知识：

| 知识点 | 说明 |
|--------|------|
| **RAG 流程** | 文档处理 → 检索 → 生成 |
| **Prompt 模板** | 上下文 + 问题的格式化 |
| **查询优化** | 查询重写、HyDE |
| **混合检索** | 提升检索准确率 |
| **重排序** | 进一步优化结果 |
| **系统评估** | 检索指标 + 生成质量 |

**下一步**: L22 将学习 API 服务开发！